In [0]:
spark

In [0]:
# Replace placeholders
storage_account = "testclient_id = "test1"
tenant_id = "test1"
client_secret = "test1"
# If no secret scope (temporary/testing only):
# client_secret = "<CLIENT_SECRET>"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
csv_path = "abfss://olistdata@olist21datastorage.dfs.core.windows.net/bronze/olist_customers_dataset.csv"
customer_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(csv_path)
display(customer_df)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
base_path = "abfss://olistdata@olist21datastorage.dfs.core.windows.net/bronze/"

customers_path = base_path + "olist_customers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
items_path = base_path + "olist_order_items_dataset.csv"
payments_path = base_path + "olist_order_payments_dataset.csv"
reviews_path = base_path + "olist_order_reviews_dataset.csv"
orders_path = base_path + "olist_orders_dataset.csv"
products_path = base_path + "olist_products_dataset.csv"
sellers_path = base_path + "olist_sellers_dataset.csv"


customer_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(customers_path)
geolocation_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(geolocation_path)
items_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(items_path)
payments_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(payments_path)
reviews_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(reviews_path)
orders_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(orders_path)
products_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(products_path)
sellers_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(sellers_path)


In [0]:
geolocation_df.display()

#Reading Data from MongoDb

In [0]:
 import pymongo

In [0]:
# importing module
from pymongo import MongoClient
import pandas as pd

hostname = "ci7hjv.h.filess.io"
database = "olistData_finallygot"
port = "61004"
username = "olistData_finallygot"
password = "032be7ead971c9951280825d8f52a1b6f14e2712"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [0]:
mydatabase

In [0]:
collection = mydatabase['product_categories']

mongo_data = pd.DataFrame(list(collection.find()))

mongo_data.columns

In [0]:

mongo_data.head()

### Cleaning the data

In [0]:
from pyspark.sql.functions import *


In [0]:
def clean_dataframe(df, name):
    print("Cleaning " + name)
    return df.dropDuplicates() \
        .dropna('all') 
        
orders_df = clean_dataframe(orders_df, "orders_df")
display(orders_df)

In [0]:
customers_df = clean_dataframe(customer_df, "customers_df")
display(customers_df)
geolocation_df = clean_dataframe(geolocation_df, "geolocation_df")
display(geolocation_df)
items_df = clean_dataframe(items_df, "items_df")
display(items_df)
payments_df = clean_dataframe(payments_df, "payments_df")
display(payments_df)
products_df = clean_dataframe(products_df, "products_df")
display(products_df)
reviews_df = clean_dataframe(reviews_df, "reviews_df")
display(reviews_df)
sellers_df = clean_dataframe(sellers_df, "sellers_df")
display(sellers_df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7553658060241257>, line 15
     13 sellers_df = clean_dataframe(sellers_df, "sellers_df")
     14 display(sellers_df)
---> 15 order_items_df = clean_da

NameError: name 'clean_da' is not defined

In [0]:

display(orders_df)

In [0]:
# convert date Columns

orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss"))
orders_df = orders_df.withColumn("order_approved_at", to_date(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss"))
orders_df = orders_df.withColumn("order_delivered_carrier_date", to_date(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss"))
orders_df = orders_df.withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss"))
orders_df = orders_df.withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss"))
display(orders_df)

In [0]:
orders_df = orders_df.withColumn("actual_delivery_time", date_diff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))

orders_df = orders_df.withColumn("estimated_delivery_time", date_diff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))

orders_df = orders_df.withColumn("delay", col("actual_delivery_time") - col("estimated_delivery_time"))

orders_df.display()

## Joining

In [0]:
orders_customers_df = orders_df.join(customer_df, "customer_id", "left")

orders_payments_df = orders_customers_df.join(payments_df, "order_id", "left")

orders_items_df = orders_payments_df.join(items_df, "order_id", "left")

orders_products_df = orders_items_df.join(products_df, "product_id", "left")

final_df = orders_products_df.join(sellers_df, "seller_id", "left")

In [0]:
final_df.display()

In [0]:
mongo_data = mongo_data.drop(columns=['_id'])

mongo_spark_df = spark.createDataFrame(mongo_data)

final_df = final_df.join(mongo_spark_df, "product_category_name", "left")

display(final_df)

In [0]:
final_df = final_df.toDF(*[f"{col}_{i}" if final_df.columns.count(col) > 1 else col for i, col in enumerate(final_df.columns)])
final_df.columns

In [0]:
final_df.display()

In [0]:
final_df.write.mode("overwrite").parquet("abfss://olistdata@olist21datastorage.dfs.core.windows.net/silver")

In [0]:
final_df.display()